In [95]:
import json
import math
from pathlib import Path
import pickle

import imageio.v3 as iio
import joblib
#import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pacmap
import pandas as pd
import piecewise_regression
from PIL import Image
from scipy import signal
from scipy.spatial.distance import euclidean
from scipy.spatial import distance_matrix
from scipy import stats
import seaborn as sns
from sklearn.feature_selection import f_regression
from sklearn.metrics.pairwise import cosine_similarity
import umap

from lib.pose_drawing import pad_and_excerpt_image, draw_armatures
from mime_db import MimeDb

# from pose_functions import *

MAX_PIXEL_MOVEMENT = (
    500  # Disregard movelets with > this many pixels movement (probably necessary)
)
MAX_3D_MOVEMENT = 10  # in meters/second, any more is probably noise

db = await MimeDb.create()
newsreels = await db.get_available_videos()
all_newsreels = sorted([nr['video_name'] for nr in newsreels])

print("All newsreels:", all_newsreels)

All newsreels: ['1980_01_13601_upscaled.mp4', '1980_02_13602_upscaled.mp4', '1980_03_13603_upscaled.mp4', '1980_04_13604_upscaled.mp4', '1980_05_13605_upscaled.mp4', '1980_06_13606_upscaled.mp4', '1980_07_13607_upscaled.mp4', '1980_08_13608_upscaled.mp4', '1980_09_13609_upscaled.mp4', '1980_10_13610_upscaled.mp4', '1980_11_13611_upscaled.mp4', '1980_12_13612_upscaled.mp4', '1980_13_13613_upscaled.mp4', '1980_14_13614_upscaled.mp4', '1980_15_13615_upscaled.mp4', '1980_16_13616_upscaled.mp4', '1980_17_13617_upscaled.mp4', '1980_18_13618_upscaled.mp4', '1980_19_13619_upscaled.mp4', '1980_20_13620_upscaled.mp4', '1980_21_13621_upscaled.mp4', '1980_22_13622_upscaled.mp4', '1980_23_13623_upscaled.mp4', '1980_24_13624_upscaled.mp4', '1980_25_13625_upscaled.mp4', '1980_26_13626_upscaled.mp4', '1980_27_13627_upscaled.mp4', '1980_28_13628_upscaled.mp4', '1980_29_13629_upscaled.mp4', '1980_30_13630_upscaled.mp4', '1980_31_13631_upscaled.mp4', '1980_32_13632_upscaled.mp4', '1980_33_13633_upscaled.

In [88]:
# ANALYSIS HELPER FUNCTIONS

FRAME_SCALE = 1
POSE_SCALE = 1

UPSCALE = 6

def get_distribution_stats(distrib, plot=False):

    if len(distrib) == 0:
        return {
            "count": 0,
            "mean": 0,
            "median": 0,
            "stdev": 0,
            "skewness": 0,
            "kurtosis": 0,
        }

    if stats.skewtest(distrib).pvalue < 0.05:
        skewness = stats.skew(distrib)
    else:
        skewness = 0

    if stats.kurtosistest(distrib).pvalue < 0.05:
        kurtosis_value = stats.kurtosis(distrib)
    else:
        kurtosis_value = 0

    if plot is True:
        plt.hist(distrib, bins="auto")  # arguments are passed to np.histogram
        plt.show()

    return {
        "count": len(distrib),
        "mean": np.mean(distrib),
        "median": np.median(distrib),
        "stdev": np.std(distrib),
        "skewness": skewness,
        "kurtosis": kurtosis_value,
    }


def get_body_units(keypoints3d, camera):
    keypoints3d = keypoints3d.to_numpy()
    return get_body_unit(project_pose(keypoints3d, camera))


def get_body_unit(pose_proj):
    shoulders = euclidean(pose_proj[1], pose_proj[2])
    left_upper_arm = euclidean(pose_proj[1], pose_proj[3])
    right_upper_arm = euclidean(pose_proj[2], pose_proj[4])
    left_forearm = euclidean(pose_proj[3], pose_proj[5])
    right_forearm = euclidean(pose_proj[4], pose_proj[6])
    hips = euclidean(pose_proj[7], pose_proj[8])
    left_thigh = euclidean(pose_proj[7], pose_proj[9])
    right_thigh = euclidean(pose_proj[8], pose_proj[10])
    left_lower_leg = euclidean(pose_proj[9], pose_proj[11])
    right_lower_leg = euclidean(pose_proj[10], pose_proj[12])

    return np.mean(
        [
            shoulders,
            left_upper_arm,
            right_upper_arm,
            left_forearm,
            right_forearm,
            hips,
            left_thigh,
            right_thigh,
            left_lower_leg,
            right_lower_leg,
        ]
    )


def project_pose(keypoints3d, camera):
    keypoints3d = keypoints3d.to_numpy()
    kp_array = np.array(np.split(keypoints3d, 13))
    kp_camera = np.array(camera)

    return kp_array + kp_camera.T


def project_pose_4dh(keypoints3d, camera):
    keypoints3d = keypoints3d.to_numpy()
    kp_array = np.array(np.split(keypoints3d, 45))
    kp_camera = np.array(camera)
    return kp_array + kp_camera.T
    #return keypoints3d + camera.T


def get_projected_pose_centroid(keypoints3d, camera):
    #keypoints3d = keypoints3d.to_numpy()
    # proj_centroid = np.mean(kp_array, axis=0) + kp_camera.T
    proj_centroid = np.mean(project_pose(keypoints3d, camera), axis=0)

    proj_centroid[2] = proj_centroid[2] / 200

    return proj_centroid


def get_point_distance_from_camera(proj_centroid):
    return math.sqrt((proj_centroid[0]**2)+(proj_centroid[1]**2)+(proj_centroid[2]**2))


def get_projected_pose_bbox_4dh(keypoints3d, camera):
    ppose = project_pose_4dh(keypoints3d, camera)
    ppose[:, 2] /= 200
    mins = np.min(ppose, axis=0).tolist()
    maxes = np.max(ppose, axis=0).tolist()

    return [mins, maxes]


def get_projected_pose_bbox(keypoints3d, camera):
    ppose = project_pose(keypoints3d, camera)
    ppose[:, 2] /= 200
    mins = np.min(ppose, axis=0).tolist()
    maxes = np.max(ppose, axis=0).tolist()

    return [mins, maxes]


def are_overlapping(bbox1, bbox2):
    # Each bbox is [[xmin, ymin, zmin], [xmax, ymax, zmax]]

    return (
        (
            (bbox1[1][0] >= bbox2[0][0] and bbox1[1][0] <= bbox2[1][0])
            or (bbox1[0][0] <= bbox2[1][0] and bbox1[0][0] >= bbox2[0][0])
        )
        and (
            (bbox1[1][1] >= bbox2[0][1] and bbox1[1][1] <= bbox2[1][1])
            or (bbox1[0][1] <= bbox2[1][1] and bbox1[0][1] >= bbox2[0][1])
        )
        and (
            (bbox1[1][2] >= bbox2[0][2] and bbox1[1][2] <= bbox2[1][2])
            or (bbox1[0][2] <= bbox2[1][2] and bbox1[0][2] >= bbox2[0][2])
        )
    )


def get_sidereal_motion(movelet, poses_df, video_fps):
    # There shouldn't ever be more than one pose centroid that matches these queries (?)
    movelet_start = poses_df[
        (poses_df["track_id"] == movelet["track_id"])
        & (poses_df["frame"] == movelet["start_frame"])
    ]

    movelet_end = poses_df[
        (poses_df["track_id"] == movelet["track_id"])
        & (poses_df["frame"] == movelet["end_frame"])
    ]

    start_centroid = movelet_start["centroid_3d"].iloc[0]
    end_centroid = movelet_end["centroid_3d"].iloc[0]

    start_time = movelet_start["frame"].iloc[0] / video_fps
    end_time = movelet_end["frame"].iloc[0] / video_fps

    time_elapsed = end_time - start_time

    if time_elapsed == 0:
        return 0
    
    speed = euclidean(start_centroid, end_centroid) / time_elapsed

    if speed > MAX_3D_MOVEMENT: # Faster than Usain Bolt (m/s), probably noise
        return 0

    return speed


def get_sidereal_mvt(movelets_df, poses_df):
    nonnull_movelets_df = movelets_df.copy()
    nonnull_movelets_df["movement"] = nonnull_movelets_df["movement"].fillna(-1)
    poses_df = poses_df.merge(shots_df, left_on="frame", right_on="frame")

    sidereal_movelets_df = nonnull_movelets_df.copy()
    sidereal_movelets_df = sidereal_movelets_df[
        sidereal_movelets_df["movement3d"] <= MAX_3D_MOVEMENT
    ]

    poses_df["centroid_3d"] = poses_df.apply(
        lambda x: get_projected_pose_centroid(x.keypoints3d, x.camera), axis=1
    )

    sidereal_movelets_df["sidereal"] = sidereal_movelets_df.apply(
        lambda x: get_sidereal_motion(x, poses_df, video_fps), axis=1
    )

    # Sidereal motion per second, based on movelet duration

    sidereal_motion_stats = get_distribution_stats(
        sidereal_movelets_df[sidereal_movelets_df["sidereal"] <= MAX_3D_MOVEMENT][
            "sidereal"
        ].tolist(),
        True,
    )
    return sidereal_motion_stats


def get_corr_mvt(movelets_df, sidereal_movelets_df, video_fps, plot=False):
    start_frames = movelets_df.start_frame.unique()
    frame_joint_motions = {} # rotational velocities
    all_motion_correlations = []
    all_speed_correlations = []
    all_sidereal_variances = []
    frame_joint_motion_corr = {}
    frame_joint_speeds = {} # absolute value of each joint's rotational velocity
    frame_joint_speed_corr = {}
    frame_sidereal_motion_var = {}
    frame_sidereal_speeds = {}
    for start_frame in start_frames:
        end_frames = movelets_df[movelets_df["start_frame"] == start_frame]["end_frame"].unique()
        if len(end_frames) == 0:
            continue
        for end_frame in end_frames:
            if end_frame <= start_frame:
                continue
            frame_diff = end_frame - start_frame
            time_elapsed = frame_diff / video_fps
            #if mvt_type == "joint_motion3d":
            joint_motions = movelets_df[(movelets_df["start_frame"] == start_frame) & (movelets_df["end_frame"] == end_frame)]["joint_motion3d"].values
        
            #else: # This is effectively 2D motion from the camera's POV
            #joint_motions = np.array(movelets_df[(movelets_df["start_frame"] == start_frame) & (movelets_df["end_frame"] == end_frame)]["norm"].values) - np.array(movelets_df[(movelets_df["start_frame"] == start_frame) & (movelets_df["end_frame"] == end_frame)]["prev_norm"].values)
            for joint_motion in joint_motions:
                if joint_motion is None:
                    continue
                joint_motion = joint_motion.to_numpy()
                #total_joint_speed = np.sum(np.abs(joint_motion))
                joint_speeds = np.abs(joint_motion)
                joint_velocities = joint_motion / time_elapsed

                if len(joint_velocities.nonzero()[0]) > 0:
                    for f in range(start_frame, end_frame+1):
                        if f not in frame_joint_motions:
                            frame_joint_motions[f] = [joint_velocities]
                            frame_joint_speeds[f] = [joint_speeds]
                        else:
                            frame_joint_motions[f].append(joint_velocities)
                            frame_joint_speeds[f].append(joint_speeds)

            sidereal_dfs = sidereal_movelets_df[(sidereal_movelets_df["start_frame"] == start_frame) & (sidereal_movelets_df["end_frame"] == end_frame)]

            if len(sidereal_dfs) > 1:
                sidereal_motions = sidereal_dfs["sidereal"].values
                if len(sidereal_motions.nonzero()[0]) > 0:
                    for f in range(start_frame, end_frame+1):
                        if f not in frame_sidereal_speeds:
                            frame_sidereal_speeds[f] = sidereal_motions
                        else:
                            frame_sidereal_speeds[f] = np.concatenate((frame_sidereal_speeds[f], sidereal_motions))
    
    for frame in sorted(frame_joint_motions.keys()):
        if len(frame_joint_motions[frame]) <= 1:
            continue
        velocity_correlations = cosine_similarity(np.array(frame_joint_motions[frame])) # np.corrcoef(frame_joint_motions[frame])
        speed_correlations = cosine_similarity(np.array(frame_joint_speeds[frame]))
        corrs_to_avg = []
        for corr in velocity_correlations[0][1:]:
            corrs_to_avg.append(corr)
            all_motion_correlations.append(corr)
        frame_joint_motion_corr[frame] = np.mean(corrs_to_avg)
        corrs_to_avg = []
        for corr in speed_correlations[0][1:]:
            corrs_to_avg.append(corr)
            all_speed_correlations.append(corr)
        frame_joint_speed_corr[frame] = np.mean(corrs_to_avg)

    for frame in sorted(frame_sidereal_speeds.keys()):
        if len(frame_sidereal_speeds[frame]) <= 1:
            continue
        frame_sidereal_variance = np.var(frame_sidereal_speeds[frame])
        frame_sidereal_motion_var[frame] = frame_sidereal_variance
        all_sidereal_variances.append(frame_sidereal_variance)
    
    if plot is True:
        corr_series_x = []
        corr_series_y = []
        for f in range(0, max(max(frame_sidereal_motion_var.keys()), video_frame_count)):
            corr_series_x.append(f)
            if f in frame_sidereal_motion_var:
                corr_series_y.append(frame_sidereal_motion_var[f])
            else:
                corr_series_y.append(0)

        plt.plot(corr_series_x, corr_series_y)
        plt.title("Variance of simultaneous sidereal motions")
        plt.show()

    if plot is True:
        corr_series_x = []
        corr_series_y = []
        for f in range(0, max(max(frame_joint_motion_corr.keys()), video_frame_count)):
            corr_series_x.append(f)
            if f in frame_joint_motion_corr:
                corr_series_y.append(frame_joint_motion_corr[f])
            else:
                corr_series_y.append(0)

        plt.plot(corr_series_x, corr_series_y)
        plt.title("Joint velocity correlations")
        plt.show()

    if plot is True:
        corr_series_x = []
        corr_series_y = []
        for f in range(0, max(max(frame_joint_speed_corr.keys()), video_frame_count)):
            corr_series_x.append(f)
            if f in frame_joint_speed_corr:
                corr_series_y.append(frame_joint_speed_corr[f])
            else:
                corr_series_y.append(0)

        plt.plot(corr_series_x, corr_series_y)
        plt.title("Joint speed correlations")
        plt.show()

    #print("Length of all motion correlations list", len(all_motion_correlations))
    motion_dist_stats = get_distribution_stats(all_motion_correlations)

    #print("Length of all speed correlations list", len(all_speed_correlations))
    speed_dist_stats = get_distribution_stats(all_speed_correlations)
    #print("SPEED DIST STATS IN FUNCTION:", speed_dist_stats)

    sidereal_variance_stats = get_distribution_stats(all_sidereal_variances)

    return motion_dist_stats, speed_dist_stats, sidereal_variance_stats


# Posture analysis:
# Differentiate standing vs seated poses (knees bent -- skip)
# Standing = avg ankle-knee-hip angle > 7/8 pi, > 9/8 pi
# For standing, get avg knee-hip-shoulder angle (first)

def get_shot_poses_and_midpoints(shots_df, poses_df):
    shot_pose_dfs = []
    shot_midpoints = [] # frames
    current_shot = -1
    first_shot_frame = 0
    last_shot_frame = 0
    current_shot_poses = []

    for row in shots_df.itertuples():
        if current_shot == -1:
            current_shot_poses.append(row)
            current_shot = row.shot
            first_shot_frame = row.frame
            continue
        elif row.shot > current_shot or row.Index == len(shots_df)-1:
            last_shot_frame = row.frame - 1
            shot_poses = poses_df[poses_df["frame"].isin(range(first_shot_frame, last_shot_frame+1))]
            shot_midframe = round((last_shot_frame - first_shot_frame) / 2)
            if len(shot_poses) > 0:
                closest_pose = shot_poses.iloc[0]
                closest_dist = -1
                for pose in shot_poses.itertuples():
                    dist = abs(pose.frame - shot_midframe)
                    if closest_dist == -1 or (dist < closest_dist):
                        closest_dist = dist
                        closest_pose = pose
                shot_midpoints.append(closest_pose.frame)
            first_shot_frame = row.frame
            current_shot = row.shot
            if len(current_shot_poses) > 0:
                shot_pose_dfs.append(pd.DataFrame(current_shot_poses, columns=current_shot_poses[0]._fields))
            current_shot_poses = []
            current_shot_poses.append(row)
        else:
            current_shot_poses.append(row)

    return shot_pose_dfs, shot_midpoints


def get_posture_stats(all_pose_angles):
    not_standing = 0
    standing = 0
    hunched = 0
    avg_waist_angle = 0
    hunched_pct = 0
    standing_waist_angles = []
    #print("max angle", (9 * math.pi / 8))
    #print("min angle", (7 * math.pi / 8))
    all_neck_angles = []
    all_waist_angles = []

    all_pose_angles = [angles.to_numpy() for angles in all_pose_angles]

    for pose in all_pose_angles:
        if pose is None: # Not sure why this is happening...
            continue

        # Neck "bent"-ness is the average of the left hip-left shoulder-nose
        # and right hip-right shoulder-nose angles
        avg_neck_angle = (pose[1] + pose[2]) / 2
        all_neck_angles.append(avg_neck_angle * 180 / math.pi)

        # Waist/back "slouch"-ness is the average of the left knee-left hip-left shoulder
        # and right kneww-right hip-right shoulder angles
        avg_waist_angle = (pose[7] + pose[8]) / 2
        all_waist_angles.append(avg_waist_angle * 180 / math.pi)

        avg_knee_angle = (pose[11] + pose[12]) / 2
        #print(avg_knee_angle, )
        # Hunched = standing only, knees not significantly bent, waist bent
        if avg_knee_angle < (9 * math.pi / 8) and avg_knee_angle > (7 * math.pi / 8):
            standing += 1
    
            standing_waist_angles.append(avg_waist_angle * 180 / math.pi)
            if avg_waist_angle < (7 * math.pi / 8):
                hunched += 1
        else:
            not_standing += 1

    if standing > 0:
        hunched_pct = hunched / standing
    
    print(standing, "standing poses out of", len(all_pose_angles), "of which hunched:", hunched, round(hunched_pct*100,2))
    
    return round(hunched_pct*100,2), standing, not_standing, standing_waist_angles, all_waist_angles, all_neck_angles


def get_all_posture_stats(poses_df, by_shot=False):
    # Determine standing: average angles[11] and angles[12]

    # Leftover code from experiments with shot midpoints and averages
    #if shots_df is not None:
    #    shot_pose_dfs, shot_midpoints = get_shot_poses_and_midpoints(poses_df)
    # if shot_midpoints is not None:
    #     midpoint_poses_df = poses_df[poses_df["frame"].isin(shot_midpoints)]
    #     all_pose_angles = midpoint_poses_df["coco13_angles3d"].tolist()
    # else:
    #     all_pose_angles = poses_df["coco13_angles3d"].tolist()

    hunched_pcts = []
    standing_pcts = []
    standing_waist_angles = []
    all_waist_angles = []
    all_neck_angles = []

    if by_shot is True:
        poses_by_shot = poses_df.groupby("shot")

        for shot, shot_poses_df in poses_by_shot:
            all_pose_angles = shot_poses_df["coco13_angles3d"].tolist()
        
            hunched_pct, standing, not_standing, stand_waist_angles, waist_angles, neck_angles = get_posture_stats(all_pose_angles)
            hunched_pcts.append(hunched_pcts)
            standing_waist_angles.extend(stand_waist_angles)
            standing_pct = round(((standing / (standing + not_standing))*100),2)
            standing_pcts.append(standing_pct)
            all_waist_angles.append(waist_angles)
            all_neck_angles.append(neck_angles)
    else:
        all_pose_angles =  poses_df["coco13_angles3d"].tolist()
        hunched_pct, standing, not_standing, stand_waist_angles, waist_angles, neck_angles = get_posture_stats(all_pose_angles)
        hunched_pcts.append(hunched_pct)
        standing_waist_angles.extend(stand_waist_angles)
        standing_pct = round(((standing / (standing + not_standing))*100),2)
        standing_pcts.append(standing_pct)
        all_waist_angles.append(waist_angles)
        all_neck_angles.append(neck_angles)

    return hunched_pcts, standing_pcts, standing, not_standing, standing_waist_angles, all_waist_angles, all_neck_angles


def pitch_yaw_to_direction_vector(pitch: float, yaw: float) -> np.ndarray:
    """
    Convert pitch and yaw angles (in radians) into a 3D unit direction vector [dx, dy, dz]
    in world space
    """
    dx = np.cos(pitch) * np.sin(yaw)
    dy = np.sin(pitch)
    dz = np.cos(pitch) * np.cos(yaw)
    
    vector = np.array([dx, dy, dz])
    # Normalize to ensure it's a unit vector
    return vector / np.linalg.norm(vector)


def estimate_gaze_from_landmarks(nose_3d, left_shoulder_3d, right_shoulder_3d):
    """
    Estimate combined 3D gaze origin and direction vector given 3D coordinates.
    """
    # Compute shoulder center midpoint as gaze origin
    shoulder_center = (left_shoulder_3d + right_shoulder_3d) / 2.0
    
    # Vector from nose to eye center provides head/face orientation reference
    face_vector = shoulder_center - nose_3d
    face_vector /= np.linalg.norm(face_vector)
    
    # Extract rough pitch and yaw from the reference orientation vector
    yaw = np.arctan2(face_vector[0], face_vector[2])
    pitch = np.arcsin(-face_vector[1])
    
    gaze_direction = pitch_yaw_to_direction_vector(pitch, yaw)
    
    return shoulder_center, gaze_direction


def estimate_orientation_from_landmarks(left_shoulder_3d, right_shoulder_3d):
    """
    Estimate 3D body orientation/direction vector given 3D coordinates.
    """

    dx = right_shoulder_3d[0] - left_shoulder_3d[0]
    dy = right_shoulder_3d[1] - left_shoulder_3d[1]

    if dx == 0 and dy == 0:
        # Any horizontal vector is orthogonal to a vertical line
        return np.array([1.0, 0.0, 0.0])
    
    # Perpendicular vector in 2D shifted to 3D horizontal plane
    ortho_vector = np.array([-dy, dx, 0.0])
    
    # Optional: Normalize the vector to make it a unit vector
    return ortho_vector / np.linalg.norm(ortho_vector)


# Plot/draw most similar poses to the average embedding
async def get_frame_and_highlight_pose(video_id, frame, keypoints, bbox, other_bbox=None, image_upscale=1):

    keypoints = np.array([round(kp * POSE_SCALE) for kp in keypoints]).reshape((13, 3))
    keypoints[:,2] = 1
    xmin = int(np.rint(np.min(keypoints[:, 0])))
    ymin = int(np.rint(np.min(keypoints[:, 1])))
    xmax = int(np.rint(np.max(keypoints[:, 0])))
    ymax = int(np.rint(np.max(keypoints[:, 1])))

    video = await db.get_video_by_id(video_id)
    video_path = f"/videos/{video['video_name'].replace('_upscaled', '')}"
    frame_img = iio.imread(video_path, index=frame - 1, plugin="pyav")

    frame_img = Image.fromarray(np.uint8(frame_img)).convert("RGB")

    frame_img = frame_img.resize((round(frame_img.width * image_upscale), round(frame_img.height * image_upscale)))

    overlay_img = ImageDraw.Draw(frame_img)

    overlay_img = draw_armatures(keypoints, overlay_img, None, upscale=FRAME_SCALE, line_width=4)

    overlay_img.rectangle(
        [(xmin * FRAME_SCALE, ymin * FRAME_SCALE), (xmax * FRAME_SCALE, ymax * FRAME_SCALE)],
        outline="purple",
        width=1 * image_upscale,
    )

    overlay_img.rectangle(
        [(bbox[0], bbox[1]), (bbox[2], bbox[3])],
        outline="red",
        width=2,
    )

    if other_bbox is not None:
        overlay_img.rectangle(
            [(other_bbox[0], other_bbox[1]), (other_bbox[2], other_bbox[3])],
            outline="orange",
            width=2,
        )

    return frame_img


async def get_cropped_pose_on_image(video_id, frame, keypoints, bbox=None, image_upscale=1):

    keypoints = np.array([round(kp * POSE_SCALE) for kp in keypoints]).reshape((13, 3))

    if bbox is None:
        xmin = round(np.rint(np.min(keypoints[:, 0])))
        ymin = round(np.rint(np.min(keypoints[:, 1])))
        xmax = round(np.rint(np.max(keypoints[:, 0])))
        ymax = round(np.rint(np.max(keypoints[:, 1])))
    else:
        xmin, ymin, xmax, ymax = [round(bbox[0]), round(bbox[1]), round(bbox[2]), round(bbox[3])]

    width = xmax - xmin
    height = ymax - ymin

    keypoints[:,2] = 1

    video = await db.get_video_by_id(video_id)
    video_path = f"/videos/{video['video_name'].replace('_upscaled', '')}"
    frame_img = iio.imread(video_path, index=frame - 1, plugin="pyav")

    orig_img = Image.fromarray(np.uint8(frame_img)).convert("RGB")

    frame_img = np.asarray(orig_img.resize((round(orig_img.width * image_upscale), round(orig_img.height * image_upscale))))

    excerpted_img = pad_and_excerpt_image(frame_img, xmin, ymin, width, height)

    excerpted_img = Image.fromarray(np.uint8(excerpted_img)).convert("RGB")

    excerpted_img = excerpted_img.resize((width * FRAME_SCALE, height * FRAME_SCALE))

    overlay_img = ImageDraw.Draw(excerpted_img)

    overlay_img = draw_armatures(keypoints, overlay_img, None, xmin, ymin, upscale=FRAME_SCALE, line_width=4)

    return excerpted_img  


async def get_video_stats(video_file, plot=False):

    video_path = Path("videos", video_file)

    # Connect to the database
    
    # Get video metadata
    video_name = video_path.name
    video_id = await db.get_video_id(video_name)

    video_data = await db.get_video_by_id(video_id)
    video_fps = video_data["fps"]
    video_frame_count = video_data["frame_count"]

    video_seconds = video_frame_count / video_fps

    #print("GETTING MOVELET DATA")

    video_movelets = await db.get_movelet_data_from_video(video_id)
    movelets_df = pd.DataFrame.from_records(
        video_movelets, columns=video_movelets[0].keys()
    )

    #print("GETTING POSE DATA")

    video_poses = await db.get_pose_data_from_video(video_id)

    #print("GETTING SHOTS DATA")

    video_shots = await db.get_video_shots(video_id)
    shots_df = pd.DataFrame.from_records(video_shots, columns=video_shots[0].keys())

    poses_df = pd.DataFrame.from_records(video_poses, columns=video_poses[0].keys())

    #print("TOTAL POSES:", len(poses_df))
    #print("TOTAL SHOTS:", len(shots_df))

    # video_3d_coco13_globals = np.array(
    #     poses_df["global3d_coco13"].to_list()
    # )  # DataFrame
    # mean_3d_coco13_global = video_3d_coco13_globals.mean(axis=0).tolist()  # ndarray
    # median_3d_coco13_global = np.median(
    #     video_3d_coco13_globals, axis=0
    # ).tolist()  # ndarray

    # video_poem_embeddings = np.array(poses_df["poem_embedding"].to_list())  # DataFrame
    # mean_poem_embedding = video_poem_embeddings.mean(axis=0).tolist()  # ndarray
    # median_poem_embedding = np.median(video_poem_embeddings, axis=0).tolist()  # ndarray

    # video_action_embeddings = np.array(poses_df["ava_action"].to_list())  # DataFrame
    # mean_action_embedding = video_action_embeddings.mean(axis=0).tolist()  # ndarray
    # median_action_embedding = np.median(
    #     video_action_embeddings, axis=0
    # ).tolist()  # ndarray

    # print("TOTAL MOVELETS:", len(movelets_df))
    # print("NON-MOTION MOVELETS:", len(movelets_df[movelets_df["movement"].isna()]))

    # print("NON-3D-MOTION MOVELETS:", len(movelets_df[movelets_df["movement3d"].isna()]))

    # print("MOVELETS WITH STILL MOTION:", len(movelets_df[movelets_df["movement"] == 0]))

    # print(
    #     "MOVELETS WITH STILL 3D MOTION:",
    #     len(movelets_df[movelets_df["movement3d"] == 0]),
    # )

    # print(
    #     "MOVELETS WITH MOVEMENT < 10px/sec:",
    #     len(
    #         movelets_df[(movelets_df["movement"] >= 0) & (movelets_df["movement"] < 10)]
    #     ),
    # )

    # print(
    #     "MEAN MOVEMENT PER MOVELET (norm px/sec):", np.nanmean(movelets_df["movement"])
    # )
    # print(
    #     "MEDIAN MOVEMENT PER MOVELET (norm px/sec):",
    #     np.nanmedian(movelets_df["movement"]),
    # )

    # print(
    #     "MEAN 3D MOVEMENT PER MOVELET (meters/sec):",
    #     np.nanmean(movelets_df["movement3d"]),
    # )
    # print(
    #     "MEDIAN 3D MOVEMENT PER MOVELET (meters/sec):",
    #     np.nanmedian(movelets_df["movement3d"]),
    # )

    nonnull_movelets_df = movelets_df.copy()
    nonnull_movelets_df["movement"] = nonnull_movelets_df["movement"].fillna(-1)

    movement_distribution = nonnull_movelets_df[
        nonnull_movelets_df["movement"] <= MAX_PIXEL_MOVEMENT
    ]["movement"]

    n, bins, patches = plt.hist(movement_distribution, bins=300)
    top_bin = n[1:].argmax()
    # print(
    #     "most frequent 2D pixel movement bin: ("
    #     + str(bins[top_bin])
    #     + ","
    #     + str(bins[top_bin + 1])
    #     + ")"
    # )
    movement_mode = (bins[top_bin] + bins[top_bin + 1]) / 2
    #print("movement mode: " + str(movement_mode))
    if plot is True:
        plt.xlabel("Movement (normalized pixels/sec)")
        plt.ylabel("# Movelets")
        plt.show()

    movement_distribution_3d = nonnull_movelets_df[
        nonnull_movelets_df["movement3d"] <= MAX_3D_MOVEMENT
    ]["movement3d"]

    n, bins, patches = plt.hist(movement_distribution_3d, bins=300)
    top_bin = n[1:].argmax()
    # print(
    #     "most frequent 3D movement bin: ("
    #     + str(bins[top_bin])
    #     + ","
    #     + str(bins[top_bin + 1])
    #     + ")"
    # )
    movement_mode_3d = (bins[top_bin] + bins[top_bin + 1]) / 2
    #print("3D movement mode: " + str((movement_mode_3d)))
    if plot is True:
        plt.xlabel("3D Movement (meters/sec)")
        plt.ylabel("# Movelets")
        plt.show()

    # movelet_poem_embeds_by_video[video_name] = (
    #     nonnull_movelets_df.poem_embedding.to_list()
    # )

    # global_3d_coco_13_by_video[video_name] = video_3d_coco13_globals

    # action_embeds_by_video[video_name] = video_action_embeddings

    # Maybe use frozen_poses to compare pose distributions? Or just use all poses in the performance?
    # frozen_movelets = movelets_df[(movelets_df['movement'] >= 0) & (movelets_df['movement'] < movement_mode)].reset_index()
    # frozen_poses = frozen_movelets['norm'].tolist()

    poses_df = poses_df.merge(shots_df, left_on="frame", right_on="frame")

    #print("LENGTH OF poses_df MERGED WITH shots_df ON frame", len(poses_df))

    poses_by_frame = poses_df.groupby("frame")

    #print("LENGTH OF poses_by_frame", len(poses_by_frame))

    all_distances = []
    all_distances_from_camera = []
    all_mean_distances_per_frame = []
    all_mean_distances_from_camera_per_frame = []
    poses_per_populated_frame = []
    overlaps_in_frame = []
    pose_similarities_per_frame = []
    pose_gaze_similarities_per_frame = []
    pose_orient_similarities_per_frame = []

    for frame, frame_poses_df in poses_by_frame:

        # print("FRAME", frame, "SHOT", frame_poses_df.iloc[0]["shot"], "POSES", len(frame_poses_df))

        poses_per_populated_frame.append(len(frame_poses_df))

        if len(frame_poses_df) <= 1:
            continue

        pose_gaze_vectors = []
        pose_orient_vectors = []
        pose_keypoints3d = frame_poses_df["global3d_coco13"]
        pose_cameras = frame_poses_df["camera"].to_list()
        for p, pose_kpts3d in enumerate(pose_keypoints3d):
            projected_pose = project_pose(pose_kpts3d, pose_cameras[p])
            pose_origin, direction_vector = estimate_gaze_from_landmarks(projected_pose[0], projected_pose[1], projected_pose[2])
            pose_gaze_vectors.append(direction_vector)
            orientation_vector = estimate_orientation_from_landmarks(projected_pose[1], projected_pose[2])
            pose_orient_vectors.append(orientation_vector)

        gaze_similarities = cosine_similarity(np.array(pose_gaze_vectors))
        pose_gaze_similarities_per_frame.append(np.mean(gaze_similarities[0][1:]))

        orient_similarities = cosine_similarity(np.array(pose_orient_vectors))
        pose_orient_similarities_per_frame.append(np.mean(orient_similarities[0][1:]))

        pose_angles = [vector.to_numpy() for vector in frame_poses_df["coco13_angles3d"]]
        pose_similarities = cosine_similarity(np.array(pose_angles))
        pose_similarities_per_frame.append(np.mean(pose_similarities[0][1:]))

        pose_centroids = frame_poses_df.apply(
            lambda x: get_projected_pose_centroid(x.keypoints3d, x.camera), axis=1
        )
        # print("POSE CENTROIDS")
        pose_centroids = [list(centroid) for centroid in pose_centroids]
        # print(pose_centroids)

        dist_matrix = distance_matrix(pose_centroids, pose_centroids)

        # print("DISTANCE MATRIX")
        # print(dist_matrix)

        upper_diagonal = list(dist_matrix[np.triu_indices(len(dist_matrix), k=1)])

        # print("UPPER DIAGONAL")
        # print(upper_diagonal)

        mean_interpose_distance = np.mean(upper_diagonal)

        all_distances.extend(upper_diagonal)
        all_mean_distances_per_frame.append(mean_interpose_distance)

        distances_from_camera = [get_point_distance_from_camera(centroid) for centroid in pose_centroids]
        all_distances_from_camera.extend(distances_from_camera)

        all_mean_distances_from_camera_per_frame.append(np.mean(distances_from_camera))

        # if len(pose_centroids) > 1:
        #   print("EUCLIDEAN")
        #   print(euclidean(pose_centroids[0], pose_centroids[1]))

        pose_bboxes = frame_poses_df.apply(
            lambda x: get_projected_pose_bbox(x.keypoints3d, x.camera), axis=1
        )
        pose_bboxes = [list(bbox) for bbox in pose_bboxes]

        # print("POSE BOUNDING BOXES")
        # print(pose_bboxes)

        total_overlaps = 0
        for i, bbox1 in enumerate(pose_bboxes):
            for j, bbox2 in enumerate(pose_bboxes):
                if i >= j:
                    continue
                if are_overlapping(bbox1, bbox2):
                    # print("OVERLAP DETECTED IN FRAME", frame_poses_df.iloc[0]["frame"], "SHOT", frame_poses_df.iloc[0]["shot"], bbox1, bbox2)
                    total_overlaps += 1

        if total_overlaps > 1:
            overlaps_in_frame.append(total_overlaps)

        # pose_body_units = shot_poses_df.apply(lambda x: get_body_units(x.keypoints3d, x.camera), axis=1)
        # print("POSE BODY UNITS")
        # print(pose_body_units)

        # print(shot_poses_df["camera"]) # camera pos is relative to *each* detection
        # if shot > 101:
        #    break

    #print("POSES PER POPULATED FRAME:")
    poses_ppf_stats = get_distribution_stats(poses_per_populated_frame, False)
    #print(poses_ppf_stats)

    #print("OVERLAPS (# POSES OVERLAPPING):")
    pose_overlaps_stats = get_distribution_stats(overlaps_in_frame, False)
    #print(pose_overlaps_stats)

    #print("MEAN INTERPOSE DISTANCES PER POPULATED FRAME:")
    interpose_dist_ppf_stats = get_distribution_stats(
        all_mean_distances_per_frame, False
    )
    #print(interpose_dist_ppf_stats)

    #print("ALL INTERPOSE DISTANCES:")
    interpose_dist_stats = get_distribution_stats(all_distances, False)
    #print(interpose_dist_stats)

    distances_from_camera_stats = get_distribution_stats(all_distances_from_camera, False)
    distances_from_camera_ppf_stats = get_distribution_stats(all_mean_distances_from_camera_per_frame, False)

    pose_similarity_stats = get_distribution_stats(pose_similarities_per_frame, False)

    # Merge non-null movelets with poses to calculate sidereal movement per pose per frame|second

    sidereal_movelets_df = nonnull_movelets_df.copy()
    sidereal_movelets_df = sidereal_movelets_df[
        sidereal_movelets_df["movement3d"] <= MAX_3D_MOVEMENT
    ]

    #print("movelets below max limit for sidereal movement:", len(sidereal_movelets_df))

    poses_df["centroid_3d"] = poses_df.apply(
        lambda x: get_projected_pose_centroid(x.keypoints3d, x.camera), axis=1
    )

    sidereal_movelets_df["sidereal"] = sidereal_movelets_df.apply(
        lambda x: get_sidereal_motion(x, poses_df, video_fps), axis=1
    )

    # Sidereal motion per second, based on movelet duration

    #print("SIDEREAL MOTION PER MOVELET:")
    sidereal_motion_stats = get_distribution_stats(
        sidereal_movelets_df[sidereal_movelets_df["sidereal"] <= MAX_3D_MOVEMENT][
            "sidereal"
        ].tolist(),
        False,
    )
    #print(sidereal_motion_stats) 

    # XXX Maybe also do intra-frame pose similarity in global 3D (over time)?

    # Stats on simultaneous motion: per-frame avg cosine similarities of directional joint rotation
    # and absolute joint rotation, per-frame variances of sidereal motion speeds (any direction)
    #motion_dist_stats, speed_dist_stats, sidereal_var_stats = get_corr_mvt(movelets_df, poses_df, shots_df, video_fps)
    #print("GETTING CORRELATED MOTION STATS")
    #motion_dist_stats, speed_dist_stats, sidereal_var_stats = get_corr_mvt(nonnull_movelets_df, sidereal_movelets_df, video_fps)

    #print("JOINT MOTION VELOCITY STATS:", motion_dist_stats)
    #print("JOIN MOTION SPEED STATS:", speed_dist_stats)

    poses_by_shot = poses_df.groupby("shot")

    tracks_per_shot = []

    for shot, shot_poses_df in poses_by_shot:
        total_tracks = shot_poses_df.track_id.unique()
        if len(total_tracks):
            tracks_per_shot.append(len(total_tracks))
        continue

    #print("TRACKS PER SHOT:")
    tracks_per_shot_stats = get_distribution_stats(tracks_per_shot, False)
    #print(tracks_per_shot_stats)

    #print("NORMALIZED MOVEMENT STATS:")
    normalized_movement_stats = get_distribution_stats(movement_distribution, False)
    #print(normalized_movement_stats)

    #print("3D MOVEMENT STATS:")
    movement_3d_stats = get_distribution_stats(movement_distribution_3d, False)
    #print(movement_3d_stats)

    pose_sims_ppf_stats = get_distribution_stats(pose_similarities_per_frame, False)

    pose_gaze_sims_ppf_stats = get_distribution_stats(pose_gaze_similarities_per_frame, False)

    pose_orient_sims_ppf_stats = get_distribution_stats(pose_orient_similarities_per_frame, False)

    video_stats = {
        "seconds": video_seconds,
        "shots": {"mean": len(video_shots)}, # Ugh
        # "mean_poem_embedding": mean_poem_embedding,
        # "median_poem_embedding": median_poem_embedding,
        # "mean_3d_global": mean_3d_coco13_global,
        # "median_3d_global": median_3d_coco13_global,
        # "mean_action_embedding": mean_action_embedding,
        # "median_action_embedding": median_action_embedding,
        "poses_ppf": poses_ppf_stats,
        "overlaps": pose_overlaps_stats,
        "interpose_dist_ppf": interpose_dist_ppf_stats,
        "interpose_dist": interpose_dist_stats,
        "distance_from_camera": distances_from_camera_stats,
        "distance_from_camera_ppf": distances_from_camera_ppf_stats,
        "sidereal": sidereal_motion_stats,
        #"sidereal_variance": sidereal_var_stats,
        "movement": normalized_movement_stats,
        "movement3d": movement_3d_stats,
        #"joint_motion_corr": motion_dist_stats,
        #"joint_speed_corr": speed_dist_stats,
        "tracks_per_shot": tracks_per_shot_stats,
        "pose_sim_ppf": pose_sims_ppf_stats,
        "pose_gaze_sim_ppf": pose_gaze_sims_ppf_stats,
        "pose_orient_sim_ppf": pose_orient_sims_ppf_stats
    }
    return video_stats

In [ ]:
# USUALLY SKIPPED: Exploratory processing of a single newsreel (for testing)

VIDEO_FILE = "1987_23_14572_upscaled.mp4"  # Just the name of the video file, no path

video_path = Path("videos", VIDEO_FILE)

# Get video metadata
video_name = video_path.name
video_id = await db.get_video_id(video_name)

video_data = await db.get_video_by_id(video_id)
video_fps = video_data["fps"]
video_frame_count = video_data["frame_count"]

video_seconds = video_frame_count / video_fps

video_movelets = await db.get_movelet_data_from_video(video_id)
movelets_df = pd.DataFrame.from_records(video_movelets, columns=video_movelets[0].keys())

video_poses = await db.get_pose_data_from_video(video_id)

video_shots = await db.get_video_shots(video_id)
shots_df = pd.DataFrame.from_records(video_shots, columns=video_shots[0].keys())

poses_df = pd.DataFrame.from_records(video_poses, columns=video_poses[0].keys())

In [ ]:
# Clustering of "held" poses for newsreels analysis
# Skip to next cell if loading data from a previous clustering run
# Skip this and the next 3 cells if only interested in pose/motion time series analysis

pose_years = []
pose_newsreels = []

first_half_held_poses = []
first_half_held_embeddings = []

second_half_held_poses = []
second_half_held_embeddings = []

for video_name in all_newsreels:
    print(video_name)

    video_year = int(video_name.split("_")[0])
    video_issue = int(video_name.split("_")[1])

    video_id = await db.get_video_id(video_name)
    
    #print("VIDEO ID", video_id)
    
    video_data = await db.get_video_by_id(video_id)
    video_fps = video_data["fps"]
    print("video_fps", video_fps)

    video_frame_count = video_data["frame_count"]
    print("Total video frames:", video_frame_count)
    
    video_seconds = video_frame_count / video_fps
    
    video_movelets = await db.get_movelet_data_from_video(video_id)
    movelets_df = pd.DataFrame.from_records(video_movelets, columns=video_movelets[0].keys())

    video_poses = await db.get_pose_data_from_video(video_id)
    poses_df = pd.DataFrame.from_records(video_poses, columns=video_poses[0].keys())

    nonnull_movelets_df = movelets_df.copy()
    #nonnull_movelets_df['movement3d'].fillna(-1, inplace=True)
    n, bins, patches = plt.hist(nonnull_movelets_df[nonnull_movelets_df['movement3d'] <= 100]['movement'], bins=300)
    plt.xlabel("Movement (normalized m/sec)")
    plt.ylabel("# Movelets")
    top_bin = n[1:].argmax()
    print('most frequent bin: (' + str(bins[top_bin]) + ',' + str(bins[top_bin+1]) + ')')
    print('mode: '+ str((bins[top_bin] + bins[top_bin+1])/2))
    movement_mode = (bins[top_bin] + bins[top_bin+1])/2

    frozen_movelets = movelets_df[(movelets_df['movement3d'] >= 0) & (movelets_df['movement3d'] < movement_mode)].reset_index()

    for movelet in frozen_movelets.to_dict("records"):
        # Prefer a target frame in the middle of the movelet, but if the actual pose
        # index is missing from this frame (which can happen sometimes), just use the
        # first frame of the movelet
        try:
            target_frame = round((movelet["end_frame"] + movelet["start_frame"]) / 2)
            target_movelet = poses_df[
                (poses_df["frame"] == target_frame)
                & (poses_df["track_id"] == movelet["track_id"])
            ]
            if len(target_movelet) == 0:
                target_frame = movelet["start_frame"]
                target_movelet = poses_df[
                    (poses_df["frame"] == target_frame)
                    & (poses_df["track_id"] == movelet["track_id"])
                ]
            target_pose = poses_df[
                (poses_df["frame"] == target_frame)
                & (poses_df["track_id"] == movelet["track_id"])
            ].iloc[0]

            frozen_embedding = target_pose['poem_embedding'].tolist()
            frozen_pose = target_pose['coco13_angles3d'].tolist()

            rounded_pose = [round(angle, 2) for angle in frozen_pose]

            pose_years.append(video_year)
            pose_newsreels.append(video_name)

            if video_year < 1985:
                first_half_held_poses.append(rounded_pose)
                first_half_held_embeddings.append(target_pose['poem_embedding'].tolist())
            else:
                second_half_held_poses.append(rounded_pose)
                second_half_held_embeddings.append(target_pose['poem_embedding'].tolist())

        except Exception:
            print("Couldn't find representative pose from movelet middle or beginning, skipping")
            match_failures += 1
            continue


joblib.dump(first_half_held_poses, "notebooks/newsreels/first_half_held_poses_rounded.pkl")
joblib.dump(second_half_held_poses, "notebooks/newsreels/second_half_held_poses_rounded.pkl")
joblib.dump(first_half_held_embeddings, "notebooks/newsreels/first_half_held_embeddings.pkl")
joblib.dump(second_half_held_embeddings, "notebooks/newsreels/second_half_held_embeddings.pkl")
json.dump({"years": pose_years, "newsreels": pose_newsreels}, open("notebooks/newsreels/held_newsreels_pose_metadata.json","w", encoding="utf-8"))

In [ ]:
# Use if loading previous clustering/embedding data
first_half_held_poses = joblib.load("notebooks/newsreels/first_half_held_poses_rounded.pkl")
second_half_held_poses = joblib.load("notebooks/newsreels/second_half_held_poses_rounded.pkl")
first_half_held_embeddings = joblib.load("notebooks/newsreels/first_half_held_embeddings.pkl")
second_half_held_embeddings = joblib.load("notebooks/newsreels/second_half_held_embeddings.pkl")
pose_metadata = json.load(open("notebooks/newsreels/held_newsreels_pose_metadata.json", "r", encoding="utf-8"))

In [ ]:
# Further held pose clustering analyses, only considering first year and last 2 years
# Also can be skipped if not interested

all_held_embeddings = first_half_held_embeddings + second_half_held_embeddings
first_year_embeds = [emb for f, emb in enumerate(all_held_embeddings) if pose_years[f] == 1980 and f % 2 == 0]
last_years_embeds = [emb for f, emb in enumerate(all_held_embeddings) if pose_years[f] >= 1989 and f & 2 == 0]

all_held_poses = first_half_held_poses + second_half_held_poses
first_year_poses = [pose for p, pose in enumerate(all_held_poses) if pose_years[p] == 1980 and p % 2 == 0]
last_years_poses = [pose for p, pose in enumerate(all_held_poses) if pose_years[p] >= 1989 and p & 2 == 0]

all_held_embeddings_first_last = first_year_embeds + last_years_embeds
print("TOTAL EMBEDDINGS:", len(all_held_embeddings_first_last))

clusterable_embedding_pacmap = pacmap.PaCMAP(
    n_components=2, n_neighbors=None, MN_ratio=0.5, FP_ratio=2.0
).fit_transform(all_held_embeddings_first_last, init="pca")

clusterable_embedding_umap = umap.UMAP(
    n_neighbors=10,
    min_dist=1.0,
    n_components=2,
    random_state=42,
    metric="cosine"
).fit_transform(all_held_embeddings_first_last)

plt.figure(figsize=(10, 10))
plt.scatter(clusterable_embedding_umap[:len(first_year_embeds), 0], clusterable_embedding_umap[:len(first_year_embeds), 1], s=1, c="red", alpha=0.1, label="red: 1980")
plt.scatter(clusterable_embedding_umap[len(first_year_embeds):, 0], clusterable_embeddin_umap[len(first_year_embeds):, 1], s=1, c="blue", alpha=0.1, label="blue: 1989-90")
ax = plt.gca()
ax.legend()
ax.grid(True)
plt.savefig("notebooks/newsreels/visualizations/UMAP_cosine.png", dpi=300)

plt.figure(figsize=(10, 10))
plt.scatter(clusterable_embedding_pacmap[:len(first_year_embeds), 0], clusterable_embedding_pacmap[:len(first_year_embeds), 1], s=1, c="red", alpha=0.1, label="red: 1980")
plt.scatter(clusterable_embedding_pacmap[len(first_year_embeds):, 0], clusterable_embedding_pacmap[len(first_year_embeds):, 1], s=1, c="blue", alpha=0.1, label="blue: 1989-90")
ax = plt.gca()
ax.legend()
ax.grid(True)
plt.savefig("notebooks/newsreels/visualizations/PaCMAP_cosine.png", dpi=300)

In [ ]:
# Very basic exploration of pose/embeddings, based on a search box in the embedding space

embedding_neighborhood = []
in_first_year = 0
in_last_years = 0
matches_in_first_year = []
matches_in_last_years = []

proj_bbox = [7, 8, 15, 16]

for e, emb in enumerate(clusterable_embedding_umap.tolist()):
    if emb[0] > proj_bbox[0] and emb[0] < proj_bbox[1] and emb[1] > proj_bbox[2] and emb[1] < proj_bbox[3]:
        if e > len(first_year_embeds):
            in_last_years += 1
            matches_in_last_years.append(all_held_embeddings_first_last[e])
        else:
            in_first_year += 1
            matches_in_first_year.append(all_held_embeddings_first_last[e])

print("IN FIRST YEAR:", in_first_year, "IN LAST YEAR:", in_last_years)

average_match = np.average(matches_in_last_years, axis=0)
print(average_match)

# Find the most similar poses to the average match
pose_subquery = "TRUE"

query = f"""
            SELECT pose.video_id, video.video_name, pose.frame, pose.pose_idx, pose.bbox, pose.norm, pose.keypoints, pose.keypoints4dh, pose.camera, poem_embedding <=> '{average_match.tolist()}' AS distance, frame.shot AS shot FROM pose, frame, video
            WHERE {pose_subquery} AND video.id = pose.video_id AND frame.video_id = pose.video_id AND pose.frame = frame.frame ORDER BY distance
            LIMIT 500
        """

similar_poses = await db._pool.fetch(query)

for pose in similar_poses[:20]:

    pose_video_name = pose.get("video_name")
    if not (pose_video_name.startswith("1980") or pose_video_name.startswith("1989") or pose_video_name.startswith("1990")):
        continue

    pose3d = pose.get("keypoints4dh").to_numpy()
    kp_array = np.array(np.split(pose3d, 45))

    xmin = np.min(kp_array[:, 0])
    xmax = np.max(kp_array[:, 0])
    ymin = np.min(kp_array[:, 1])
    ymax = np.max(kp_array[:, 1])
    bbox_4dh = [xmin, ymin, xmax, ymax]

    visible_bbox = pose.get("bbox")
    visible_bbox[2] = visible_bbox[0] + visible_bbox[2]
    visible_bbox[3] = visible_bbox[1] + visible_bbox[3]

    max_bbox = [min(xmin, visible_bbox[0]), min(ymin, visible_bbox[1]), max(xmax, visible_bbox[2]), max(ymax, visible_bbox[3])]

    pose_image = await get_cropped_pose_on_image(
        pose.get("video_id"), pose.get("frame"), pose.get("keypoints").to_numpy(), max_bbox, image_upscale=6
    )
    frame_image = await get_frame_and_highlight_pose(
        pose.get("video_id"), pose.get("frame"), pose.get("keypoints").to_numpy(), visible_bbox, max_bbox, image_upscale=6
    )

    print(
        pose.get("video_name"), "frame", pose.get("frame"), "pose", pose.get("pose_idx")
    )
    fig, ax = plt.subplots()
    plt.axis("off")
    plt.imshow(frame_image)
    plt.show()

In [89]:
# Newsreels analysis, all videos: pose motion, slouch, spacing, orientation analysis
# Skip to next cell to load data from a previous run

nr_corr_mvt = {}
nr_posture = { "pct_hunched": {}, "pct_standing": {}, "standing": {}, "not_standing": {}, "avg_waist_angle": {}, "avg_standing_waist_angle": {}, "avg_neck_angle": {}}

# nr_pct_hunched = {}
# nr_pct_standing = {}
# nr_standing = {}
# nr_not_standing = {}
# nr_avg_waist_angle = {}

for video_name in all_newsreels:
    print(video_name)

    video_id = await db.get_video_id(video_name)
    
    #print("VIDEO ID", video_id)
    
    video_data = await db.get_video_by_id(video_id)
    video_fps = video_data["fps"]

    video_frame_count = video_data["frame_count"]
    
    video_seconds = video_frame_count / video_fps
    
    video_movelets = await db.get_movelet_data_from_video(video_id)

    video_poses = await db.get_pose_data_from_video(video_id)
    
    video_shots = await db.get_video_shots(video_id)
    shots_df = pd.DataFrame.from_records(video_shots, columns=video_shots[0].keys())

    if len(video_poses) == 0:
        print("EMPTY POSES FOR", video_name)
        nr_posture["pct_hunched"][video_name] = [0]
    else:
        poses_df = pd.DataFrame.from_records(video_poses, columns=video_poses[0].keys())
        poses_df = poses_df.merge(shots_df, left_on="frame", right_on="frame")
        hunched_pcts, standing_pcts, standing, not_standing, standing_waist_angles, all_waist_angles, all_neck_angles = get_all_posture_stats(poses_df)
        nr_posture["pct_hunched"][video_name] = hunched_pcts
        nr_posture["pct_standing"][video_name] = standing_pcts
        nr_posture["standing"][video_name] = standing
        nr_posture["not_standing"][video_name] = not_standing
        nr_posture["avg_standing_waist_angle"][video_name] = np.mean(standing_waist_angles)
        nr_posture["avg_waist_angle"][video_name] = np.mean(all_waist_angles)
        nr_posture["avg_neck_angle"][video_name] = np.mean(all_neck_angles)
        # Better to fail on any error, rather than filling in empty values
        # except Exception as e:
        #    print("PROBLEM WITH POSES FOR", video_name, video_id, e)
        #    nr_pct_hunched[video_name] = [0]
        #    nr_pct_standing[video_name] = [0]
        #    nr_standing[video_name] = [0]
        #    nr_not_standing[video_name] = [0]
        #    nr_avg_waist_angle[video_name] = 0

    if len(video_movelets) == 0:
        print("PROBLEM WITH MOVELETS FOR", video_name)
        nr_corr_mvt[video_name] = { "joint_rotation_correlation": 0, "joint_speed_correlation": 0, "sidereal_motion_variance": 0}
    else:
        movelets_df = pd.DataFrame.from_records(video_movelets, columns=video_movelets[0].keys())
        nonnull_movelets_df = movelets_df.copy()
        nonnull_movelets_df["movement"] = nonnull_movelets_df["movement"].fillna(-1)

        sidereal_movelets_df = nonnull_movelets_df.copy()
        sidereal_movelets_df = sidereal_movelets_df[
            sidereal_movelets_df["movement3d"] <= MAX_3D_MOVEMENT
        ]

        poses_df["centroid_3d"] = poses_df.apply(
            lambda x: get_projected_pose_centroid(x.keypoints3d, x.camera), axis=1
        )

        sidereal_movelets_df["sidereal"] = sidereal_movelets_df.apply(
            lambda x: get_sidereal_motion(x, poses_df, video_fps), axis=1
        )

        sidereal_motion_stats = get_distribution_stats(
            sidereal_movelets_df[sidereal_movelets_df["sidereal"] <= MAX_3D_MOVEMENT][
                "sidereal"
            ].tolist(),
            False,
        )

        motion_dist_stats, speed_dist_stats, sidereal_variance_stats = get_corr_mvt(nonnull_movelets_df, sidereal_movelets_df, video_fps)
        nr_corr_mvt[video_name] = { "joint_rotation_correlation": motion_dist_stats["mean"], "joint_speed_correlation": speed_dist_stats["mean"], "sidereal_motion_variance": sidereal_variance_stats["mean"], "sidereal_motion": sidereal_motion_stats["mean"]}

pickle.dump(nr_corr_mvt, open("notebooks/newsreels/nr_corr_mvt.pkl", "wb"))
json.dump(nr_posture, open("notebooks/newsreels/nr_posture.json", "w", encoding="utf-8"))

# json.dump(nr_pct_hunched, open("notebooks/newsreels/nr_pct_hunched.json", "w", encoding="utf-8"))
# json.dump(nr_standing, open("notebooks/newsreels/nr_standing.json", "w", encoding="utf-8"))
# json.dump(nr_not_standing, open("notebooks/newsreels/nr_not_standing.json", "w", encoding="utf-8"))
# json.dump(nr_pct_standing, open("notebooks/newsreels/nr_pct_standing.json", "w", encoding="utf-8"))

1980_01_13601_upscaled.mp4
1998 standing poses out of 23193 of which hunched: 125 6.26
1980_02_13602_upscaled.mp4
4957 standing poses out of 33054 of which hunched: 180 3.63
1980_03_13603_upscaled.mp4
3168 standing poses out of 29888 of which hunched: 203 6.41
1980_04_13604_upscaled.mp4
1606 standing poses out of 28336 of which hunched: 192 11.96
1980_05_13605_upscaled.mp4
1524 standing poses out of 32413 of which hunched: 297 19.49
1980_06_13606_upscaled.mp4
2402 standing poses out of 42658 of which hunched: 99 4.12
1980_07_13607_upscaled.mp4
1947 standing poses out of 30264 of which hunched: 143 7.34
1980_08_13608_upscaled.mp4
1617 standing poses out of 43590 of which hunched: 5 0.31
1980_09_13609_upscaled.mp4
3735 standing poses out of 43055 of which hunched: 146 3.91
1980_10_13610_upscaled.mp4
5070 standing poses out of 41489 of which hunched: 145 2.86
1980_11_13611_upscaled.mp4
2824 standing poses out of 41272 of which hunched: 140 4.96
1980_12_13612_upscaled.mp4
1352 standing pos

In [ ]:
# Run if reloading newsreels pose data from a previous run

nr_corr_mvt = pickle.load(open("notebooks/newsreels/nr_corr_mvt.pkl", "rb"))
nr_posture = json.load(open("notebooks/newsreels/nr_posture.json", "r", encoding="utf-8"))

# nr_pct_hunched = json.load(open("notebooks/newsreels/nr_pct_hunched.json", "r", encoding="utf-8"))
# nr_standing = json.load(open("notebooks/newsreels/nr_standing.json", "r", encoding="utf-8"))
# nr_not_standing = json.load(open("notebooks/newsreels/nr_not_standing.json", "r", encoding="utf-8"))
# nr_pct_standing = json.load(open("notebooks/newsreels/nr_pct_standing.json", "r", encoding="utf-8"))

In [ ]:
# Print some basic pose/motion/posture data reports

print("issue\t\t\t\tmean_corr_rot")

for video_name in nr_corr_mvt:
    print(nr_corr_mvt[video_name]['joint_speed_correlation'])
    if nr_corr_mvt[video_name]['joint_speed_correlation'] > .06:
        data = [video_name, f"{round(float(nr_corr_mvt[video_name]['joint_rotation_correlation']),5)}"]
        print("\t".join(data))

print("issue\t\t\t\tpct_hunched\tposes\tstanding\thunched")

for video_name in nr_posture["pct_hunched"]:
    if nr_posture["pct_hunched"][video_name][0] < 1:
        data = [video_name, f"{nr_posture['pct_hunched'][video_name][0]}%", "", str(nr_posture['standing'][video_name] + nr_posture['not_standing'][video_name]), str(nr_posture['standing'][video_name]), "", str(round(nr_posture['pct_hunched'][video_name][0]/100 * nr_posture['standing'][video_name]))]
        print("\t".join(data))

In [ ]:
# NEWSREELS ANALYSIS - calculate all high-level distance, motion, orientation data
# Skip to next frame to reload data from a previous run

# From https://stackoverflow.com/questions/50916422/python-typeerror-object-of-type-int64-is-not-json-serializable
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

motion_stats_per_video = {}

for video_name in all_newsreels:
    print("GETTING MOTION STATS FOR", video_name)
    motion_stats_per_video[video_name] = await get_video_stats(video_name, plot=False)

    json.dump(
        motion_stats_per_video, open("notebooks/newsreels/newsreel_motion_stats.json", "w", encoding="utf-8"), cls=NpEncoder, indent=4
    )
    

In [ ]:
# Run to reload distance, motion, orientation data from a prior run
motion_stats_per_video = json.load(open("notebooks/newsreels/newsreel_motion_stats.json", "r", encoding="utf-8"))

In [ ]:
# Report on a statistic for all videos

stat_name = "interpose_dist"
print("issue\t\t\t\t\tinterpose_dist_mean\t\tmedian\t\t\tdev\t\t\tskewness")
for video_name in motion_stats_per_video:
    if not video_name.startswith("1984"):
        continue
    if motion_stats_per_video[video_name]["interpose_dist"]["mean"] < 1.1:
        data = [video_name, motion_stats_per_video[video_name][stat_name]["mean"], motion_stats_per_video[video_name][stat_name]["median"], motion_stats_per_video[video_name][stat_name]["stdev"], motion_stats_per_video[video_name][stat_name]["skewness"]]
        print("\t\t\t".join([str(round(item, 5)) if type(item) != str else str(item) for item in data]))

In [94]:
# NEWSREEL ANALYSIS - PLOT AVG MOTION/DIST STATS FOR ALL VIDEOS

# Cram the correlated movement stats into the motion stats data structure,
# for more convenient plotting code
for video_id in nr_corr_mvt:
    for metric_name in nr_corr_mvt[video_id]:
        # It's not actually the mean value in many cases, but we'll let this pass
        motion_stats_per_video[video_id][metric_name] = { "mean": nr_corr_mvt[video_id][metric_name] }

    motion_stats_per_video[video_id]["pct_hunched"] = {"mean": nr_posture["pct_hunched"][video_id][0]}
    motion_stats_per_video[video_id]["standing"] = {"mean": nr_posture["standing"][video_id]}
    motion_stats_per_video[video_id]["total_poses"] = {"mean": nr_posture["standing"][video_id] + nr_posture["not_standing"][video_id]}
    motion_stats_per_video[video_id]["pct_standing"] = {"mean": nr_posture["pct_standing"][video_id][0]}
    # XXX Remove radians-to-degrees calculations after updating the data struct
    motion_stats_per_video[video_id]["avg_waist_angle"] = {"mean": nr_posture["avg_waist_angle"][video_id]}
    motion_stats_per_video[video_id]["avg_standing_waist_angle"] = {"mean": nr_posture["avg_standing_waist_angle"][video_id]}
    motion_stats_per_video[video_id]["avg_neck_angle"] = {"mean": nr_posture["avg_neck_angle"][video_id]}

all_metrics = []
pct_hunched_y = []

first_video = next(iter(motion_stats_per_video)) 
metric_slugs = [metric for metric in motion_stats_per_video[first_video]]

metric_title = {"shots": "Total shots per newsreel",
                "pct_hunched": "Percent of standing poses that are hunched",
                "total_poses": "Total poses detected per newsreel",
                "standing": "Total standing poses per newsreel",
                "pct_standing": "Percent of poses per newsreel that are standing",
                "avg_standing_waist_angle": "Avg standing pose waist angle",
                "avg_waist_angle": "Avg pose waist angle",
                "avg_neck_angle": "Avg pose neck angle",
                "joint_rotation_correlation": "Joint rotation correlation",
                "joint_speed_correlation": "Joint speed correlation",
                "sidereal_motion_variance": "Sidereal motion variance",
                "sidereal_motion": "Sidereal motion",
                "distance_from_camera": "Mean distance from camera (m) by pose",
                "distance_from_camera_ppf": "Mean distance from camera (m) by populated frame",
                "poses_ppf": "Avg # poses per populated frame", 
                "overlaps": "Avg # overlapping poses per frame", 
                "interpose_dist_ppf": "Avg inter-pose distance per populated frame (m)", 
                "interpose_dist": "Avg inter-pose distance per frame (m)", 
                "sidereal": "Avg sidereal motion per frame (m)", 
                "movement": "Avg 2D in-place movement per frame (pixels)", 
                "movement3d": "3D in-place movement per frame (m)", 
                "tracks_per_shot": "Avg # tracked persons per shot",
                "pose_sim_ppf": "Avg pose similarity per populated frame",
                "pose_gaze_sim_ppf": "Avg gaze direction similarity per populated frame",
                "pose_orient_sim_ppf": "Avg pose orientation similarity per populated frame"
                }

nr_ids = [ '{0:.2f}'.format(float(n.split('_')[0] + '.' + n.split('_')[1])) for n in all_newsreels ]

reels_per_year = {}
max_reel_per_year = {}
for nr in nr_ids:
    year = int(nr.split('.')[0])
    ed = int(nr.split('.')[1])

    if year not in max_reel_per_year:
        max_reel_per_year[year] = ed
        reels_per_year[year] = [ed]
    else:
        max_reel_per_year[year] = max(max_reel_per_year[year], ed)
        reels_per_year[year].append(ed)

total_reels_per_year = { year: len(reels_per_year[year]) for year in reels_per_year}

nr_x = []
for nr in nr_ids:
    year = int(nr.split('.')[0])
    ed = int(nr.split('.')[1])
    step = (ed-1) / max_reel_per_year[year]
    nr_x.append(year + step - 1980)

xticks = []
nr_year_ticks = []
prev_year = ""
for n, nr_label in enumerate(nr_ids):
    year = nr_label.split(".")[0]
    if year != prev_year:
        xticks.append(nr_x[n])
        nr_year_ticks.append(year)
        prev_year = year

def get_metric_stats(metric, nr_x, motion_stats_per_video, pct_hunched_y=[]):

    metric_y = [motion_stats_per_video[vn][metric]["mean"] for vn in motion_stats_per_video]

    if metric == "pct_hunched":
        pct_hunched_y = metric_y

    plt.figure(figsize=(10, 10))
    stats_df = pd.DataFrame.from_dict({'year': nr_x, 'metric_data': metric_y})
    cg = sns.lmplot(x='year', y='metric_data', data=stats_df, fit_reg=True, scatter_kws={"s": 2}, line_kws={"color":"red"}, ci=95, n_boot=1000)
    cg.set(title=metric_title[metric])
    cg.set(xticks=range(0,11))
    cg.set(xticklabels=nr_year_ticks)
    cg.set_axis_labels("year", metric.replace("_", " "))
    cg.savefig(f"notebooks/newsreels/visualizations/newsreels_{metric}.png", dpi=300)
    plt.close()

    # Statistical tests

    # Computes quick univariate F-statistics and p-values for all columns in X
    #f_stats, p_values = f_regression(np.array(nr_x).reshape(-1, 1), np.array(metric_y))

    #for i, (f_stat, p) in enumerate(zip(f_stats, p_values)):
    #    print(f"{metric} -> F-statistic: {f_stat:.4f}, p-value: {p:.4e}")

    res = stats.linregress(nr_x, metric_y)

    slope = res.slope
    std_err = res.stderr
    df = len(nr_x)-2 # Degrees of freedom (N - 2 for simple linear regression)

    confidence_level = 0.95
    alpha = 1 - confidence_level
    t_crit = stats.t.ppf(1 - alpha/2, df)

    lower_ci = slope - (t_crit * std_err)
    upper_ci = slope + (t_crit * std_err)

    # Segmented regression with one breakpoint (expected to be after April 1985)
    plt.figure(figsize=(10, 10))
    pw_fit = piecewise_regression.Fit(nr_x, metric_y, n_breakpoints=1, n_boot=1000)
    pw_results = pw_fit.get_results()
    #pw_estimates = pw_results["estimates"]

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=2)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    # Consider one or both of the following for more behind-the-scenes details
    if metric == "pct_hunched":
        pw_fit.summary()
    ax = plt.gca()
    ax.set(title=f"{metric_title[metric]}: segmented fit")
    ax.set(xticks=range(0,11))
    ax.set(xticklabels=nr_year_ticks)
    plt.xlabel("year")
    plt.ylabel(metric.replace('_', ' '))
    plt.savefig(f"notebooks/newsreels/visualizations/newsreels_{metric}_piecewise.png", dpi=300)
    plt.close()

    print(metric)
    print("Piecewise:", pw_results)
    print(f"Slope: {slope:.4f}")
    print(f"Degrees of Freedom: {df}")
    print(f"Slope Standard Error: {std_err:.4f}")
    print(f"Intercept Standard Error: {res.intercept_stderr:.4f}")
    print(f"R-value: {res.rvalue:.4f}")
    print(f"P-value: {res.pvalue}")
    print(f"95% Confidence Interval: [{lower_ci:.4f}, {upper_ci:.4f}]")

    # For every metric other than pct_hunched, compute its cross correlation with pct_hunched
    if metric != "pct_hunched":
        corr_w_pct_hunched = np.corrcoef(pct_hunched_y, metric_y)[0, 1]
        print(f"Pearson correlation with pct hunched poses: {corr_w_pct_hunched:.4f}")

        x_corr = signal.correlate(pct_hunched_y, metric_y, mode='full')
        # Note that the sequences are always the same length if we're comparing per-newsreel stats
        lags = signal.correlation_lags(len(pct_hunched_y), len(metric_y), mode='full')
        best_lag_index = np.argmax(np.abs(x_corr))
        best_lag = lags[best_lag_index]
        #print(f"Cross-correlation with pct hunched poses: {x_corr}")
        print(f"Cross-correlation suggests {metric} is shifted {best_lag} issues relative to % hunched")

    print()

    return metric_y

pct_hunched_y = get_metric_stats("pct_hunched", nr_x, motion_stats_per_video)

for metric in metric_slugs:
    if metric == "seconds" or metric == "pct_hunched":
        continue
    get_metric_stats(metric, nr_x, motion_stats_per_video, pct_hunched_y)


                    Breakpoint Regression Results                     
No. Observations                      418
No. Model Parameters                    4
Degrees of Freedom                    414
Res. Sum of Squares               17966.3
Total Sum of Squares              20145.2
R Squared                        0.108159
Adjusted R Squared               0.099522
Converged:                           True
                    Estimate      Std Err            t        P>|t|       [0.025       0.975]
----------------------------------------------------------------------------------------------------
const                5.03539        0.736        6.846      2.75e-11       3.5896       6.4812
alpha1              0.421696        0.191       2.2033        0.0281     0.045481      0.79791
beta1                1.12927        0.543        2.081             -     0.062574        2.196
breakpoint1          6.41824         1.26            -             -       3.9342       8.9023
-----------------

/usr/local/lib/python3.10/site-packages/seaborn/axisgrid.py:453: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=figsize)
/tmp/ipykernel_92/2337970430.py:125: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 10))


standing
Piecewise: {'davies': 0.8968494859633879, 'estimates': {'const': {'estimate': 2513.373626373916, 'se': 680.8245450586189, 'confidence_interval': (1175.069593199779, 3851.6776595480533), 't_stat': 3.6916613018960924, 'p_t': 0.0002526038338490312}, 'beta1': {'estimate': -2632.971620460088, 'se': 4621.648653713421, 'confidence_interval': (-11717.795424287473, 6451.852183367297), 't_stat': -0.5697039774636563, 'p_t': '-'}, 'breakpoint1': {'estimate': 0.2665418069775656, 'se': 0.2873667829849607, 'confidence_interval': (-0.2983381278901429, 0.8314217418452741), 't_stat': '-', 'p_t': '-'}, 'alpha1': {'estimate': 2436.395604393516, 'se': 4621.590265048485, 'confidence_interval': (-6648.313424215581, 11521.104633002613), 't_stat': 0.5271768946760917, 'p_t': 0.5983531250512628}, 'alpha2': {'estimate': -196.57601606657227, 'se': 23.231452389468714, 'confidence_interval': (-242.24232856216832, -150.9097035709762), 't_stat': -8.461632650900645, 'p_t': 4.597788264517376e-16}}, 'bic': 6013.

/tmp/ipykernel_92/2337970430.py:93: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(10, 10))


total_poses
Piecewise: {'davies': 0.00030214340111559516, 'estimates': {'const': {'estimate': 37323.1811133411, 'se': 1241.3965718472268, 'confidence_interval': (34882.95469828339, 39763.40752839881), 't_stat': 30.065477833407698, 'p_t': 3.764723089040592e-106}, 'beta1': {'estimate': 6335.498314468897, 'se': 1817.4849380887504, 'confidence_interval': (2762.848884404655, 9908.147744533138), 't_stat': 3.4858601475571214, 'p_t': '-'}, 'breakpoint1': {'estimate': 7.974982566591773, 'se': 0.5235910220041399, 'confidence_interval': (6.94575413825775, 9.004210994925796), 't_stat': '-', 'p_t': '-'}, 'alpha1': {'estimate': -1235.691648817747, 'se': 283.7148161673915, 'confidence_interval': (-1793.3928737572605, -677.9904238782332), 't_stat': -4.355400488103835, 'p_t': 1.6765043735238836e-05}, 'alpha2': {'estimate': 5099.80666565115, 'se': 1795.204000459715, 'confidence_interval': (1570.9551111309993, 8628.6582201713), 't_stat': 2.840795065265671, 'p_t': 0.004722287791479198}}, 'bic': 7864.26429

<Figure size 1000x1000 with 0 Axes>

... (20 outputs left out)

<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

<Figure size 1000x1000 with 0 Axes>

In [ ]:
years = range(1980, 1991)

poses_by_year = {}
minutes_by_year = {}

for year in years:
    poses_by_year[year] = 0
    minutes_by_year[year] = 0

for video_id in all_newsreels:
    nr_year = int(video_id.split("_")[0])
    poses_by_year[nr_year] += (nr_posture["not_standing"][video_id] + nr_posture["standing"][video_id]) / 1000
    minutes_by_year[nr_year] += float(motion_stats_per_video[video_id]["seconds"]) / 60

poses_y = [poses_by_year[year] for year in years]
minutes_y = [minutes_by_year[year] for year in years]

plt.figure(figsize=(10, 10))
fig, ax1 = plt.subplots()

color = 'tab:orange'
ax1.set_xlabel('Year')
ax1.set_ylabel('Minutes of newsreels', color=color)
ax1.plot(years, minutes_y, color=color, linestyle='-', marker='o', label="minutes")
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0)
ax1.set_xticks(years)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('# poses (thousands)', color=color)  # we already handled the x-label with ax1
ax2.plot(years, poses_y, color=color, linestyle='-', marker='s', label="poses")
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(0)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc=0)

plt.title("Duration of newsreels and detected poses per year in study corpus")
plt.savefig("notebooks/newsreels/visualizations/newsreels_per_year.png", dpi=300)
plt.close